In [ ]:
# LOCO (Leave-One-Cohort-Out) Cross-Validation Orchestration
# Manuscript: Methods 'Leave-one-cohort-out (LOCO) validation'; Results sections 1-3 are built on its outputs.
# Per fold: 10 MDFS runs (seeds 42-51) on the pooled training cohorts, top 2% of pairs per run,
# consensus = pairs in >= 6 of 10 runs, LR/GM synthetic features, one RF per feature configuration.
# This notebook orchestrates the LOCO cross-validation loop and calls framework.ipynb for ML operations
#
# IMPORTANT: The framework functions will be loaded automatically in the next cell.
# If that fails, you can manually run framework.ipynb cells first.

import pandas as pd
import numpy as np
import warnings
import os
from collections import Counter

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

# --- CONFIGURATION ---
ALL_COHORT_STEMS = [
    'Public_study__FengQ_2015', 'Public_study__GuptaA_2019', 'Public_study__LiuNN_2022',
    'Public_study__ThomasAM_2018b', 'Public_study__VogtmannE_2016', 'Public_study__WirbelJ_2018',
    'Public_study__YachidaS_2019', 'Public_study__YangJ_2020', 'Public_study__YangY_2021',
    'Public_study__YuJ_2015', 'Public_study__ZellerG_2014', 'This_study_cohort1__ONCOBIOME_AtezoTRIBE',
    'This_study_cohort2__ONCOBIOME_COLOBIOME', 'This_study_cohort3__ONCOBIOME_IIGM_CZ',
    'This_study_cohort4__ONCOBIOME_IIGM_IT', 'This_study_cohort5__NSHII', 'This_study_cohort6__IIGM_TU'
]
TRAINING_ONLY_COHORT = 'This_study_cohort5__NSHII'
# Cohorts to leave out of every training set (empty = main analysis). Used for the sensitivity run that
# drops the single-class / training-only cohorts (Reviewer 1, comment 5): see run_loco.py --exclude-training-only
EXCLUDE_FROM_TRAINING = ()

# Framework functions should be available after running framework.ipynb
# If not available, the code will fail with a clear error message

In [ ]:
# Load framework functions from framework.ipynb
# This executes all code cells from framework.ipynb to make functions available
import json

def load_framework_notebook(notebook_path='framework.ipynb'):
    """Load and execute all code cells from a Jupyter notebook."""
    try:
        # Read the notebook as JSON
        with open(notebook_path, 'r', encoding='utf-8') as f:
            nb = json.load(f)
        
        # Execute all code cells (skip markdown cells)
        for cell in nb.get('cells', []):
            if cell.get('cell_type') == 'code':
                source = cell.get('source', [])
                if isinstance(source, list):
                    code = ''.join(source)
                else:
                    code = source
                
                if code.strip():  # Only execute non-empty cells
                    exec(code, globals())
        
        print("✓ Framework functions loaded successfully from framework.ipynb")
        return True
    except FileNotFoundError:
        print(f"⚠ Error: {notebook_path} not found. Please ensure framework.ipynb exists in the current directory.")
        return False
    except Exception as e:
        print(f"⚠ Error loading framework: {e}")
        print("Please run framework.ipynb cells manually first, then continue with this notebook.")
        return False

# Load the framework
load_framework_notebook('framework.ipynb')

In [ ]:
def load_prepared_data(X_file, y_file, data_type='functions'):
    """
    Loads already-prepared data files (from prep_data.ipynb).
    
    Parameters:
    -----------
    X_file : str
        Path to prepared feature matrix file (TSV format, samples x features)
    y_file : str
        Path to prepared labels file (TSV format)
    data_type : str
        Type of data: 'functions', 'taxa', or 'combined' (for display purposes)
    
    Returns:
    --------
    X : pd.DataFrame
        Feature matrix (samples x features)
    y : pd.Series
        Target labels (0 = CRC, 1 = Control)
    """
    try:
        # Load prepared feature matrix
        X = pd.read_csv(X_file, sep="\t", index_col=0)
        
        # Load prepared labels
        y = pd.read_csv(y_file, sep="\t", index_col=0).squeeze()
        
        # Ensure labels are integers
        y = y.astype(int)
        
        # Verify alignment
        if not X.index.equals(y.index):
            # Align if needed
            common_samples = X.index.intersection(y.index)
            X = X.loc[common_samples]
            y = y.loc[common_samples]
            print(f"  Aligned samples: {len(common_samples)} common samples")
        
        print(f"  Successfully loaded {data_type} data with {len(X)} samples and {X.shape[1]} features.")
        return X, y
    except Exception as e:
        print(f"  Error loading prepared data: {e}")
        raise

In [ ]:
def main(prepared_data_dir='prepared_data', data_type='functions', 
         use_synthetic=True, scenario='All Features + Synthetic', output_dir='results'):
    """
    Main method to perform LOCO (Leave-One-Cohort-Out) cross-validation with feature selection, 
    synthetic feature generation, and RF classification.
    
    Parameters:
    -----------
    prepared_data_dir : str
        Directory containing prepared data files (from prep_data.ipynb)
    data_type : str
        Type of data: 'functions', 'taxa', or 'combined' (for display purposes)
    use_synthetic : bool
        Whether to generate and use synthetic features (default: True)
    scenario : str
        Which scenario to evaluate: 'All Features', 'Synthetic Only', or 'All Features + Synthetic'
    output_dir : str
        Directory to save results (default: 'results')
    
    Returns:
    --------
    dict : Dictionary containing LOCO results, summary statistics, and feature counts
    """
    print("="*80)
    print(f"LOCO Analysis Pipeline - {data_type.upper()}")
    print("="*80)
    
    # Create output directory
    os.makedirs(output_dir, exist_ok=True)
    
    # 1. Load all prepared cohort data
    print("\n--- Step 1: Loading all prepared cohort data ---")
    all_cohort_data = {}
    for cohort_stem in ALL_COHORT_STEMS:
        X_file = os.path.join(prepared_data_dir, f"X_{cohort_stem}_{data_type}.tsv")
        y_file = os.path.join(prepared_data_dir, f"y_{cohort_stem}_{data_type}.tsv")
        
        if os.path.exists(X_file) and os.path.exists(y_file):
            X, y = load_prepared_data(X_file, y_file, data_type)
            all_cohort_data[cohort_stem] = (X, y)
        else:
            print(f"  Warning: Missing files for '{cohort_stem}'. Skipping.")
    
    # 2. Identify test-eligible cohorts
    print("\n--- Step 2: Identifying test-eligible cohorts ---")
    test_cohort_candidates = []
    for stem, data in all_cohort_data.items():
        if stem == TRAINING_ONLY_COHORT:
            print(f"  '{stem}' is designated for training only. Skipping.")
            continue
        if data is not None:
            _, y = data
            counts = y.value_counts()
            if counts.get(0, 0) >= 15 and counts.get(1, 0) >= 15:
                test_cohort_candidates.append(stem)
                print(f"  '{stem}' is ELIGIBLE (CRC={counts.get(0,0)}, Control={counts.get(1,0)})")
            else:
                print(f"  '{stem}' is INELIGIBLE (CRC={counts.get(0,0)}, Control={counts.get(1,0)})")
    
    print(f"\n>>> Found {len(test_cohort_candidates)} eligible cohorts for LOCO testing.\n")
    
    # 3. LOCO loop with N_RUNS_PER_FOLD MDFS runs per fold for consensus
    # Track both AUC and Accuracy (saved separately)
    loco_results_auc = []
    loco_results_acc = []
    base_feature_counts = Counter()
    synthetic_feature_counts = Counter()

    # Track IG information across all folds and runs (new format)
    all_mdfs_1d_data = []  # 1D MDFS tables (Feature, IG, p_value, adjusted_p_value) per fold and run
    all_mdfs_ig_data = []  # List of dicts: {fold, test_cohort, run, base, contributing, base_IG_1D, contributing_IG_1D, IG_2D_added, total_IG}

    N_RUNS_PER_FOLD = 10      # independent MDFS runs per fold (seeds 42..51)
    CONSENSUS_MIN_RUNS = 6    # a pair is kept if it is in the top fraction of >= 6 runs
    TOP_PAIR_FRACTION = 0.02  # top 2% of pairs by total_IG per run

    for i, test_cohort_stem in enumerate(test_cohort_candidates):
        print(f"\n{'='*80}")
        print(f"FOLD {i+1}/{len(test_cohort_candidates)}: Testing on '{test_cohort_stem}'")
        print(f"{'='*80}")
        
        # Prepare test data
        X_test_raw, y_test = all_cohort_data[test_cohort_stem]
        
        # Prepare training data (all cohorts except test)
        train_cohort_stems = [s for s in ALL_COHORT_STEMS if s != test_cohort_stem and s not in EXCLUDE_FROM_TRAINING]
        train_cohort_data_X = {name: all_cohort_data[name][0] for name in train_cohort_stems 
                            if name in all_cohort_data and all_cohort_data[name] is not None}
        train_cohort_data_y = {name: all_cohort_data[name][1] for name in train_cohort_stems 
                            if name in all_cohort_data and all_cohort_data[name] is not None}
        
        if not train_cohort_data_X:
            print(f"  No training data available. Skipping fold.")
            continue
        
        # Align features across training cohorts
        print(f"\n  Aligning training cohorts...")
        if len(train_cohort_data_X) == 0:
            print(f"  No training data available. Skipping fold.")
            continue
        
        # Find intersection of all features across training cohorts
        all_train_features = sorted(set.intersection(*[set(df.columns) for df in train_cohort_data_X.values()]))  # sorted: deterministic column order for MDFS
        
        if len(all_train_features) == 0:
            print(f"  Warning: No common features found across training cohorts. Skipping fold.")
            continue
        
        # Filter each training cohort to only common features
        train_cohort_data_X = {name: df[list(all_train_features)] for name, df in train_cohort_data_X.items()}
        
        # Concatenate all training cohorts
        X_train_raw = pd.concat(list(train_cohort_data_X.values()))
        y_train = pd.concat(list(train_cohort_data_y.values()))
        
        print(f"  Aligned to {len(all_train_features)} common features across {len(train_cohort_data_X)} training cohorts")
        print(f"  Training samples: {len(X_train_raw)}")
        
        # ===== N_RUNS_PER_FOLD INDEPENDENT MDFS RUNS PER FOLD =====
        mdfs_pairs_per_run = []  # List of sets of pairs for each run
        mdfs_ig_per_run = []  # List of dicts: {(base, contributing): total_IG} for each run
        individual_ig_per_run = []  # List of dicts: {feature: ig} for each run
        
        for run_idx in range(N_RUNS_PER_FOLD):
            print(f"\n  --- RUN {run_idx+1}/{N_RUNS_PER_FOLD} ---")
            
            # Seed for this MDFS run
            run_seed = 42 + run_idx
            
            # Generate synthetic features with IG tracking
            synthetic_feature_pairs = []
            mdfs_info = None
            if use_synthetic:
                print(f"  Generating synthetic features (run {run_idx+1})...")
                mdfs_info = run_mdfs_feature_selection(
                    X_train_raw, y_train, 
                    temp_prefix=f'temp_mdfs_{i}_run{run_idx}',
                    seed=run_seed,
                    return_ig_info=True
                )
                if mdfs_info:
                    if mdfs_info.get('ig_1d_table') is not None:
                        _t1 = mdfs_info['ig_1d_table'].copy()
                        _t1.insert(0, 'run', run_idx); _t1.insert(0, 'test_cohort', test_cohort_stem); _t1.insert(0, 'fold', i)
                        all_mdfs_1d_data.append(_t1)
                    # Keep the top TOP_PAIR_FRACTION of pairs per run (keeps the RF feature matrix
                    # tractable). Score each unordered pair by max total_IG across its two directions.
                    _all_pairs = mdfs_info['pairs']
                    _pair_ig = mdfs_info['pair_ig']
                    _pair_score = {}
                    for _p in _all_pairs:
                        _a, _b = _p
                        _pair_score[_p] = max(_pair_ig.get((_a, _b), 0.0), _pair_ig.get((_b, _a), 0.0))
                    _sorted_pairs = sorted(_pair_score.items(), key=lambda x: -x[1])
                    _n_keep = max(1, int(len(_sorted_pairs) * TOP_PAIR_FRACTION))
                    _kept = [p for p, _ in _sorted_pairs[:_n_keep]]
                    _kept_set = set(_kept)
                    mdfs_info['pairs'] = _kept
                    mdfs_info['pair_ig'] = {
                        (a, b): ig for (a, b), ig in _pair_ig.items()
                        if tuple(sorted([a, b])) in _kept_set
                    }
                    print(f"  After top-{TOP_PAIR_FRACTION:.0%} per-run filter: {len(_kept)} pairs (from {len(_all_pairs)})")

                    synthetic_feature_pairs = mdfs_info['pairs']
                    mdfs_pairs_per_run.append(set(synthetic_feature_pairs))
                    mdfs_ig_per_run.append(mdfs_info['pair_ig'])
                    individual_ig_per_run.append(mdfs_info['individual_ig'])
                    
                    # Save IG data for this run (new format: base/contributing with 1D and 2D IGs)
                    for (base, contrib), total_ig in mdfs_info['pair_ig'].items():
                        base_ig_1d = mdfs_info['individual_ig'].get(base, 0.0)
                        contrib_ig_1d = mdfs_info['individual_ig'].get(contrib, 0.0)
                        ig_2d_added = total_ig - base_ig_1d
                        all_mdfs_ig_data.append({
                            'fold': i,
                            'test_cohort': test_cohort_stem,
                            'run': run_idx,
                            'base': base,
                            'contributing': contrib,
                            'base_IG_1D': base_ig_1d,
                            'contributing_IG_1D': contrib_ig_1d,
                            'IG_2D_added': ig_2d_added,
                            'total_IG': total_ig,
                        })
        
        # ===== FIND CONSENSUS: pairs in the top fraction of >= CONSENSUS_MIN_RUNS runs =====
        print(f"\n  Finding consensus pairs (in >= {CONSENSUS_MIN_RUNS}/{N_RUNS_PER_FOLD} runs)...")
        
        if mdfs_pairs_per_run:
            pair_counts = Counter()
            for pairs_set in mdfs_pairs_per_run:
                pair_counts.update(pairs_set)
            consensus_pairs = sorted(pair for pair, count in pair_counts.items() if count >= CONSENSUS_MIN_RUNS)
            print(f"  Consensus MDFS pairs (in >= {CONSENSUS_MIN_RUNS}/{N_RUNS_PER_FOLD} runs): {len(consensus_pairs)}")
            synthetic_feature_pairs = consensus_pairs
        else:
            synthetic_feature_pairs = []
        
        
        # ===== FINAL EVALUATION WITH CONSENSUS FEATURES =====
        # Generate synthetic features using consensus pairs
        X_synthetic_train = generate_synthetic_features(X_train_raw, synthetic_feature_pairs) if synthetic_feature_pairs else pd.DataFrame(index=X_train_raw.index)
        X_test = X_test_raw.reindex(columns=X_train_raw.columns, fill_value=0)
        
        # Prepare scenarios
        scenarios = {
            'All Features': (X_train_raw, X_test),
            'Synthetic Only': (X_synthetic_train, generate_synthetic_features(X_test, synthetic_feature_pairs)) if not X_synthetic_train.empty else None,
            'All Features + Synthetic': (
                pd.concat([X_train_raw, X_synthetic_train], axis=1),
                pd.concat([X_test, generate_synthetic_features(X_test, synthetic_feature_pairs)], axis=1)
            ) if not X_synthetic_train.empty else (X_train_raw, X_test)
        }
        
        # Evaluate each scenario
        fold_aucs = {'Test Cohort': test_cohort_stem, 'N Samples': len(y_test)}
        fold_accs = {'Test Cohort': test_cohort_stem, 'N Samples': len(y_test)}
        for name, (X_tr_orig, X_te_orig) in scenarios.items():
            if X_tr_orig is None or X_te_orig is None:
                fold_aucs[name] = np.nan
                fold_accs[name] = np.nan
                continue
            
            print(f"\n  Evaluating: {name}")
            if X_tr_orig.empty or X_tr_orig.shape[1] < 10:
                fold_aucs[name] = np.nan
                fold_accs[name] = np.nan
                print(f"    Skipping: insufficient features")
                continue
            
            # Use framework.train_and_evaluate() to train and evaluate model
            result = train_and_evaluate(
                X_train=X_tr_orig,
                y_train=y_train,
                X_test=X_te_orig,
                y_test=y_test,
                k_values=TOP_K_VALUES,
                model_params=MODEL_PARAMS
            )
            
            fold_aucs[name] = result['auc']
            fold_accs[name] = result['accuracy']
            print(f"    k={result['best_k']}, AUC={result['auc']:.4f}, Accuracy={result['accuracy']:.4f}")
            
            # Track features for 'All Features + Synthetic' scenario (using consensus)
            if name == 'All Features + Synthetic':
                selected_features = result['selected_features']
                base_feature_counts.update([f for f in selected_features if not f.startswith(SYNTHETIC_PREFIXES)])
                synthetic_feature_counts.update([f for f in selected_features if f.startswith(SYNTHETIC_PREFIXES)])
        
        loco_results_auc.append(fold_aucs)
        loco_results_acc.append(fold_accs)

        # Per-fold AUC summary so progress can be reviewed live.
        print(f"\n  ===== FOLD {i+1}/{len(test_cohort_candidates)} AUC SUMMARY ({test_cohort_stem}, n={len(y_test)}) =====")
        for _name in ['All Features', 'Synthetic Only', 'All Features + Synthetic']:
            _auc = fold_aucs.get(_name, np.nan)
            _acc = fold_accs.get(_name, np.nan)
            _auc_s = f"{_auc:.4f}" if isinstance(_auc, float) and not np.isnan(_auc) else "  N/A "
            _acc_s = f"{_acc:.4f}" if isinstance(_acc, float) and not np.isnan(_acc) else "  N/A "
            print(f"    {_name:30s}  AUC={_auc_s}  Acc={_acc_s}")
        print()

    # Save IG information (new format)
    if all_mdfs_ig_data:
        ig_df = pd.DataFrame(all_mdfs_ig_data)
        ig_path = os.path.join(output_dir, f"mdfs_ig_info_{data_type}.csv")
        ig_df.to_csv(ig_path, index=False)
        print(f"\nMDFS IG information saved to: {ig_path}")
    if all_mdfs_1d_data:
        _p1 = os.path.join(output_dir, f"mdfs_1d_{data_type}.csv")
        pd.concat(all_mdfs_1d_data, ignore_index=True).to_csv(_p1, index=False)
        print(f"MDFS 1D IG and p-values (every fold and run) saved to: {_p1}")

    # 4. Final reporting
    if loco_results_auc:
        # 4. Final reporting
        if loco_results_auc:
            # ---- AUC table ----
            results_df_auc = pd.DataFrame(loco_results_auc).set_index('Test Cohort')
            model_cols_auc = [col for col in results_df_auc.columns if col != 'N Samples']
            
            mean_aucs = results_df_auc[model_cols_auc].mean()
            weighted_mean_aucs = pd.Series(
                [np.average(results_df_auc[col].dropna(), weights=results_df_auc.loc[results_df_auc[col].notna(), 'N Samples'])
                for col in model_cols_auc],
                index=model_cols_auc,
            )
            summary_df_auc = pd.DataFrame({'Mean AUC': mean_aucs, 'Weighted Mean AUC': weighted_mean_aucs})
            
            # Add summary rows
            results_df_auc.loc['Mean'] = {**{'N Samples': np.nan}, **mean_aucs.to_dict()}
            results_df_auc.loc['Weighted Mean'] = {**{'N Samples': np.nan}, **weighted_mean_aucs.to_dict()}
            results_df_auc = results_df_auc[results_df_auc.columns]

            # ---- Accuracy table ----
            results_df_acc = pd.DataFrame(loco_results_acc).set_index('Test Cohort')
            model_cols_acc = [col for col in results_df_acc.columns if col != 'N Samples']

            mean_accs = results_df_acc[model_cols_acc].mean()
            weighted_mean_accs = pd.Series(
                [np.average(results_df_acc[col].dropna(), weights=results_df_acc.loc[results_df_acc[col].notna(), 'N Samples'])
                for col in model_cols_acc],
                index=model_cols_acc,
            )
            summary_df_acc = pd.DataFrame({'Mean Accuracy': mean_accs, 'Weighted Mean Accuracy': weighted_mean_accs})

            results_df_acc.loc['Mean'] = {**{'N Samples': np.nan}, **mean_accs.to_dict()}
            results_df_acc.loc['Weighted Mean'] = {**{'N Samples': np.nan}, **weighted_mean_accs.to_dict()}
            results_df_acc = results_df_acc[results_df_acc.columns]
            
            # Feature stability analysis
            num_folds = len(test_cohort_candidates)
            base_features_df = pd.DataFrame(base_feature_counts.items(), columns=['Feature', 'Selection Count'])
            if not base_features_df.empty:
                base_features_df['Selection %'] = (base_features_df['Selection Count'] / num_folds) * 100
                base_features_df = base_features_df.sort_values(by='Selection Count', ascending=False).reset_index(drop=True)
            
            synthetic_features_df = pd.DataFrame(synthetic_feature_counts.items(), columns=['Feature', 'Selection Count'])
            if not synthetic_features_df.empty:
                synthetic_features_df['Selection %'] = (synthetic_features_df['Selection Count'] / num_folds) * 100
                synthetic_features_df = synthetic_features_df.sort_values(by='Selection Count', ascending=False).reset_index(drop=True)
            
            # Print summary
            print("\n\n" + "="*80)
            print(f"LOCO Performance Summary - {data_type.upper()}")
            print("="*80)

            print("\nAUC Performance per Test Cohort:")
            print(results_df_auc.to_string(float_format="%.4f"))
            print("\n" + "-"*80)
            print("Overall AUC Summary:")
            print(summary_df_auc.to_string(float_format="%.4f"))

            print("\n" + "-"*80)
            print("Accuracy Performance per Test Cohort:")
            print(results_df_acc.to_string(float_format="%.4f"))
            print("\n" + "-"*80)
            print("Overall Accuracy Summary:")
            print(summary_df_acc.to_string(float_format="%.4f"))
            
            if not base_features_df.empty:
                print(f"\nTop Base Features (appeared in {num_folds} folds):")
                print(base_features_df.head(20).to_string(index=False, float_format="%.1f"))
            
            if not synthetic_features_df.empty:
                print(f"\nTop Synthetic Features (appeared in {num_folds} folds):")
                print(synthetic_features_df.head(20).to_string(index=False, float_format="%.1f"))
            
            # Save results
            results_path = os.path.join(output_dir, f"loco_results_{data_type}.csv")
            results_df_auc.to_csv(results_path)
            print(f"\nAUC results saved to: {results_path}")

            acc_path = os.path.join(output_dir, f"loco_accuracy_{data_type}.csv")
            results_df_acc.to_csv(acc_path)
            print(f"Accuracy results saved to: {acc_path}")
            
            if not base_features_df.empty:
                base_features_path = os.path.join(output_dir, f"base_features_{data_type}.csv")
                base_features_df.to_csv(base_features_path, index=False)
                print(f"Base features saved to: {base_features_path}")
            
            if not synthetic_features_df.empty:
                synthetic_features_path = os.path.join(output_dir, f"synthetic_features_{data_type}.csv")
                synthetic_features_df.to_csv(synthetic_features_path, index=False)
                print(f"Synthetic features saved to: {synthetic_features_path}")
            
            print("="*80)
            
            return {
                'results_df_auc': results_df_auc,
                'summary_df_auc': summary_df_auc,
                'results_df_acc': results_df_acc,
                'summary_df_acc': summary_df_acc,
                'base_features': base_features_df,
                'synthetic_features': synthetic_features_df,
                'loco_results_auc': loco_results_auc,
                'loco_results_acc': loco_results_acc,
            }
        else:
            print("No results to report.")
            return None

In [ ]:
# Run the LOCO pipeline for the three data types (Methods: LOCO validation).
# Each data type takes roughly 0.5-2 h (14 folds x 10 MDFS runs, then 3 RF scenarios per fold).
# Headless alternative, one data type per process:  python run_loco.py <taxa|functions|combined>
PREPARED_DATA_DIR = '../M1_data_preparation/prepared_data'

results = {}
for dt in ['taxa', 'functions', 'combined']:
    print("\n" + "=" * 80 + f"\n{dt.upper()} ANALYSIS\n" + "=" * 80)
    results[dt] = main(
        prepared_data_dir=PREPARED_DATA_DIR,
        data_type=dt,
        use_synthetic=True,
        scenario='All Features + Synthetic',
        output_dir=f'results/{dt}',
    )
print("\nALL ANALYSES COMPLETE - results in results/taxa, results/functions, results/combined")
